# Challenge 6a: Land Cover Classification from Multispectral Satellite Data

**Author:** Dr Rob Lyon

**Version:** 1.0

## Code & License
Copyright &copy; 2026 Robert Lyon. All Rights Reserved.

Permission is granted to use, copy, and modify this material for
personal educational purposes only. Redistribution, resale, or use
in commercial or institutional settings requires prior written
permission from the author. This material is provided for educational
purposes as part of a structured course. The author accepts no
liability for incorrect results or decisions arising from use of this
material. All datasets used in this challenge are synthetic and do
not represent proprietary or confidential experimental data.

## Table of Contents

1. [Introduction](#1-introduction)
2. [Useful Links](#2-useful-links)
3. [Libraries and Environment Setup](#3-libraries-and-environment-setup)
   - [3.1 Working Environment](#31-working-environment)
   - [3.2 Libraries Used](#32-libraries-used)
   - [3.3 Data](#33-data)
   - [3.4 Imports](#34-imports)
4. [The Data](#4-the-data)
   - [4.1 Loading the Data](#41-loading-the-data)
   - [4.2 Understanding the Features](#42-understanding-the-features)
   - [4.3 Exploring the Data](#43-exploring-the-data)
   - [4.4 Preprocessing](#44-preprocessing)
5. [The Challenge](#5-the-challenge)
   - [5.1 Training the Model](#51-training-the-model)
   - [5.2 Evaluating the Model](#52-evaluating-the-model)
   - [5.3 Interpretation](#53-interpretation)
   - [5.4 Reflection Questions](#54-reflection-questions)
6. [Solution](#6-solution)
7. [References](#7-references)

---
## 1. Introduction

Every land cover map you have seen was produced by classifying satellite
imagery, and almost every policy decision about land use, flood risk,
deforestation, and urban expansion depends on the accuracy of that
classification. The European Copernicus Land Service, the USGS National
Land Cover Database, and the ESA Climate Change Initiative Land Cover
product all rest on the ability to assign a class label to each pixel in
a multispectral image. Getting it right at scale, across seasons, across
continents, and across sensor types remains one of the central problems
in applied remote sensing.

The principle is simple. Different surface types reflect and
absorb electromagnetic radiation differently. Healthy vegetation strongly
reflects near-infrared light while absorbing red. Open water absorbs nearly
everything beyond the red. Urban surfaces and bare soil have high reflectance
across the visible bands. A classifier trained on these spectral signatures
can, in principle, assign a label to every pixel without a human having to
look at it.

In practice, several things make this hard. Medium-resolution sensors
(Sentinel-2 at 10 metres, Landsat at 30 metres) routinely produce mixed
pixels: a single pixel straddles a road and a grass verge, a shoreline, or
a forest edge. Its spectral signature is a weighted average of the two
surfaces and fits neither class cleanly. Urban vegetation (parks, street
trees, garden plots) is abundant enough to push urban pixels into the NDVI
range of genuine vegetation. Turbid or shallow water has elevated red
reflectance that overlaps with dark urban materials. Wet tarmac and deep
building shadows have near-zero NIR that mimics open water. These overlaps
are not edge cases; they affect a substantial fraction of any real scene.

This challenge uses the **RobSat** dataset, a synthetic multispectral dataset
inspired by the spectral properties of Sentinel-2 imagery. It contains 15,000
pixel observations labelled as one of three land cover classes:

- **Urban** (class 0, ~60%): built surfaces including roads, rooftops, car
  parks, and bare soil. The majority class, and the noisiest, because urban
  environments contain many spectrally distinct sub-types all given the same
  label at medium resolution.
- **Vegetation** (class 1, ~30%): forest, cropland, and grassland. Generally
  well-separated from water but overlaps with urban at the sparse/mixed end
  of the canopy coverage spectrum.
- **Water** (class 2, ~10%): lakes, rivers, reservoirs, and coastal pixels.
  The minority class. Clear water is easy to identify, but turbid and
  shallow water, and dark urban surfaces, generate the most confusion.

You will train a decision tree classifier on this dataset. Decision trees
are popular in remote sensing because they are fast, interpretable, and
handle correlated features without requiring the feature independence
assumptions that Naive Bayes requires. They also have a well-known failure
mode on noisy data: without depth constraints they memorise training noise
and generalise poorly. Choosing the right tree depth, and understanding
what happens when you get it wrong, is the central skill this challenge
develops.

### Learning Objectives

- Apply `DecisionTreeClassifier` to a three-class remote sensing
  classification problem with correlated spectral features
- Understand and control tree depth as the primary mechanism for
  managing the bias-variance trade-off in decision trees
- Evaluate a multiclass classifier on an imbalanced dataset using
  per-class precision, recall, F1 score, and the confusion matrix
- Visualise a fitted decision tree and read off the decision rules
  it has learned, relating them to the physical properties of the
  spectral features
- Identify which features the tree uses most (via feature importances)
  and assess whether the ranking makes physical sense

---
## 2. Useful Links

| Resource | URL |
|---|---|
| `DecisionTreeClassifier` (scikit-learn) | https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html |
| Decision tree visualisation: `plot_tree` | https://scikit-learn.org/stable/modules/generated/sklearn.tree.plot_tree.html |
| `classification_report` (scikit-learn) | https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html |
| `ConfusionMatrixDisplay` (scikit-learn) | https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html |
| ESA Sentinel-2 spectral bands overview | https://sentinels.copernicus.eu/web/sentinel/missions/sentinel-2/instrument-payload/resolution-and-swath |
| NDVI definition and applications (NASA) | https://earthobservatory.nasa.gov/features/MeasuringVegetation/measuring_vegetation_2.php |
| McFeeters (1996) NDWI original paper | https://doi.org/10.1080/01431169608948714 |
| ESA CCI Land Cover product | https://www.esa-landcover-cci.org |

---
## 3. Libraries and Environment Setup

### 3.1 Working Environment

This notebook requires Python 3.9 or later. All libraries are part of the
standard scientific Python stack. No specialist remote sensing library is
needed; the features are pre-computed and stored in the CSV.

### 3.2 Libraries Used

| Library | Purpose |
|---|---|
| `numpy` | Numerical operations |
| `pandas` | Loading and inspecting the dataset |
| `matplotlib` | Plotting, including the tree visualisation |
| `seaborn` | Distribution plots and heatmaps |
| `sklearn.model_selection` | Stratified train/test splitting |
| `sklearn.tree` | `DecisionTreeClassifier` and `plot_tree` |
| `sklearn.metrics` | Confusion matrix, classification report, accuracy |

### 3.3 Data

| Property | Value |
|---|---|
| Filename | `robsat.csv` |
| Location | `data/robsat.csv` (relative to this notebook) |
| Total rows | 15,000 |
| Features | 8 columns (reflectances, indices, texture, thermal) |
| Target column | `land_cover` |
| Class 0 | urban: ~8,791 samples (~58.6%) |
| Class 1 | vegetation: ~4,519 samples (~30.1%) |
| Class 2 | water: ~1,690 samples (~11.3%) |

The dataset is moderately imbalanced: urban pixels outnumber water pixels
by roughly 5:1. This reflects the composition of a typical mixed
urban-rural-coastal scene captured by a medium-resolution satellite.
The imbalance means accuracy alone is a misleading headline metric:
a classifier that always predicted urban would score nearly 59%.

### 3.4 Imports

In [ ]:
# Listing 1.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

CLASS_NAMES = ['urban', 'vegetation', 'water']

print('Libraries loaded successfully.')

---
## 4. The Data

### 4.1 Loading the Data

In [ ]:
# Listing 2.
df = pd.read_csv('data/robsat.csv')

print(f'Dataset shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print()
df.head(10)

### 4.2 Understanding the Features

Eight features are stored in the dataset. Four are raw spectral reflectance
values from individual bands, two are derived spectral indices computed from
those bands, one is a spatial texture measure, and one is a thermal quantity
from a separate infrared sensor. All reflectance values are dimensionless
(surface reflectance scaled 0-1). The indices range from -1 to +1 by
construction.

| Feature | Type | Units | Description |
|---|---|---|---|
| `blue_ref` | Continuous | reflectance (0-1) | Blue band (~490 nm). Urban and bare soil reflect strongly in the blue. Vegetation absorbs blue for photosynthesis. Water scatters blue weakly. |
| `red_ref` | Continuous | reflectance (0-1) | Red band (~665 nm). Chlorophyll absorbs red strongly, so vegetation has low values. Urban and soil are high. Water is very low. |
| `nir_ref` | Continuous | reflectance (0-1) | Near-infrared (~842 nm). Leaf cell structure causes extremely high NIR reflectance in healthy vegetation. Urban is moderate. Water absorbs NIR almost completely. |
| `swir_ref` | Continuous | reflectance (0-1) | Short-wave infrared (~1610 nm). Sensitive to surface moisture. Dry urban and soil: high. Moist vegetation: moderate. Water: near zero. |
| `ndvi` | Continuous | dimensionless (-1 to 1) | Normalised Difference Vegetation Index: (NIR - Red) / (NIR + Red). High values (~0.4-0.9) indicate healthy vegetation. Urban/soil: low or slightly negative. Water: near zero or slightly negative. Derived from `nir_ref` and `red_ref`. |
| `ndwi` | Continuous | dimensionless (-1 to 1) | Normalised Difference Water Index (McFeeters 1996): (Green - NIR) / (Green + NIR). Positive values indicate water. Negative for land. Derived from a green approximation and `nir_ref`. Strongly anti-correlated with `ndvi`. |
| `texture_var` | Continuous | reflectance² | Local variance of the NIR band in a 3x3 pixel window. High values indicate spatially heterogeneous surfaces (urban roofscapes, field boundaries). Low values indicate homogeneous surfaces (open water, dense canopy). |
| `thermal_bt` | Continuous | Kelvin | Thermal infrared brightness temperature. Urban areas are warmer due to the heat island effect. Vegetation is cooler via evapotranspiration. Water is thermally buffered and stable. Largely independent of the spectral reflectance indices. |

### 4.3 Exploring the Data

In [ ]:
# Listing 3.
print('Class distribution:')
counts = df['land_cover'].value_counts().sort_index()
for label, count in counts.items():
    print(f'  Class {label} ({CLASS_NAMES[label]:12s}): {count:5d}  ({100*count/len(df):.1f}%)')

print()
print('Mean feature values by class:')
print(df.groupby('land_cover')[
    ['blue_ref', 'red_ref', 'nir_ref', 'swir_ref',
     'ndvi', 'ndwi', 'texture_var', 'thermal_bt']
].mean().round(3).to_string())

**Questions to consider:**

- Which features show the largest difference in mean values between classes?
  Based on the descriptions in Section 4.2, does this match your expectation?
- Urban and water both have low mean NDVI, yet vegetation has very high NDVI.
  Will NDVI alone be sufficient to separate all three classes, or does the
  model need additional features to resolve the urban-water boundary?
- The water class has only ~1,690 samples compared to ~8,791 urban samples.
  What effect might this imbalance have on how the decision tree is trained,
  and on which metric you should use to evaluate water class performance?

In [ ]:
# Listing 4.
# Feature distributions by class
feature_cols = ['blue_ref', 'red_ref', 'nir_ref', 'swir_ref',
                'ndvi', 'ndwi', 'texture_var', 'thermal_bt']
colours = {0: 'steelblue', 1: 'seagreen', 2: 'darkorange'}

fig, axes = plt.subplots(2, 4, figsize=(18, 7))
axes = axes.flatten()

for ax, feat in zip(axes, feature_cols):
    for cls in [0, 1, 2]:
        subset = df.loc[df['land_cover'] == cls, feat]
        subset.plot(kind='kde', ax=ax, label=CLASS_NAMES[cls],
                    color=colours[cls], linewidth=1.8)
    ax.set_title(feat, fontsize=10)
    ax.set_xlabel('')
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions by Land Cover Class (RobSat)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

**Questions to consider:**

- The `ndvi` and `ndwi` distributions show very clear class separation for
  vegetation, but urban and water overlap considerably in both. Which other
  features help resolve this ambiguity?
- Look at `thermal_bt`. The urban distribution is clearly shifted to higher
  temperatures compared to vegetation and water, but vegetation and water
  have similar means. How much discriminating power does thermal add for
  the vegetation-water boundary?
- The urban class has wide, relatively flat distributions for most features.
  This reflects real spectral heterogeneity: roads, rooftops, bare soil, and
  urban parks all get the same label. How do you expect this to affect the
  decision tree's ability to classify urban pixels cleanly?

In [ ]:
# Listing 5.
# Correlation matrix of all features
corr = df.drop(columns='land_cover').corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='coolwarm',
    vmin=-1, vmax=1, linewidths=0.5, ax=ax, square=True
)
ax.set_title('Feature Correlation Matrix (RobSat)', fontsize=13)
plt.tight_layout()
plt.show()

**Questions to consider:**

- `ndvi` and `ndwi` are very strongly negatively correlated (around -0.95).
  Explain why: what are both indices computed from, and which shared variable
  drives the near-perfect anti-correlation?
- `thermal_bt` correlates moderately with the visible band reflectances
  (around 0.5-0.6) but weakly with NIR and the NIR-derived indices. Why?
  Which surface type drives the correlation with visible bands, and why does
  it not extend to NIR?
- `texture_var` correlates with `swir_ref` and `red_ref` but weakly with
  `nir_ref` and the indices. Given the physical meaning of texture in this
  context, does this make sense?
- Decision trees do not assume feature independence and are not harmed by
  correlated features in the way logistic regression coefficients can be.
  However, correlated features do affect feature importances. If `ndvi` and
  `ndwi` carry almost identical information, what do you expect to happen
  to their individual importance scores?

### 4.4 Preprocessing

Decision trees split on individual feature thresholds and are therefore
invariant to monotonic rescaling. You do not need to apply `StandardScaler`
before fitting a `DecisionTreeClassifier`: scaling the features by any
positive constant does not change which threshold the tree chooses at each
node. This is a genuine advantage over gradient-based methods like logistic
regression.

What you do need to handle carefully is the train/test split. The dataset
has a 60:30:10 class distribution. Without stratification, a random split
could leave the water class significantly under-represented in either the
training or test set, leading to unstable per-class metrics. Stratification
preserves the class ratios in both splits.

In [ ]:
# Listing 6.
# TODO: Implement preprocessing.
#
# 1. Separate features (X) from the target column ('land_cover') (y).
#    Use all eight feature columns.
#
# 2. Split into training and test sets.
#    Use an 80/20 split, random_state=42, and stratify by y.
#    With 15,000 samples this gives 3,000 test samples: enough for
#    stable per-class metrics even for the minority water class.
#
# 3. Print the shapes of X_train, X_test, y_train, y_test.
#    Also print the class counts in y_train and y_test to confirm
#    that stratification preserved the 60:30:10 ratio.
#
# Note: no feature scaling is needed here. Explain in a comment
# why this is the case for decision trees.

# YOUR CODE HERE

---
## 5. The Challenge

Your task is to train a `DecisionTreeClassifier` to assign land cover
labels to RobSat pixels, and to find a tree depth that generalises well
to unseen data without overfitting the training noise.

Decision trees grow by recursively splitting the training set on the
feature threshold that most reduces impurity (Gini impurity by default
in scikit-learn). Without any constraint on depth, the tree will keep
splitting until every leaf contains only one class, perfectly fitting
the training data and memorising the label noise. On test data, this
produces poor generalisation: the tree has learned the noise rather
than the signal.

As covered in Notebook_6, `max_depth` is the primary control for this.
Your job is to find a depth that is neither too shallow (high bias,
misses real structure) nor too deep (high variance, memorises noise).

### 5.1 Training the Model

In [ ]:
# Listing 7.
# TODO: Train a decision tree classifier.
#
# Part A: Train with no depth constraint first.
# 1. Instantiate DecisionTreeClassifier with random_state=42 and no
#    max_depth limit.
# 2. Fit to the training data.
# 3. Report training accuracy and test accuracy.
#    The gap between them is the overfitting signal.
#
# Part B: Find a good max_depth.
# 4. Loop over max_depth values from 1 to 15.
#    For each depth, train a tree, record training accuracy and
#    test accuracy. Store both.
# 5. Plot training accuracy and test accuracy against max_depth
#    on the same axes.
#    Where does test accuracy peak? Where does the gap between
#    training and test accuracy become large?
#
# Part C: Train the final model.
# 6. Instantiate and fit a DecisionTreeClassifier with the depth you
#    identified as optimal in Part B. Use random_state=42.
#    This is the model you will evaluate and interpret in Sections 5.2
#    and 5.3.

# YOUR CODE HERE

### 5.2 Evaluating the Model

With a 60:30:10 class distribution, accuracy is a poor headline metric.
A naive classifier that always predicted urban would achieve ~59% on the
test set. The classification report breaks performance down by class
and shows where the model is failing.

For land cover mapping, the operationally important question is often:
how often does the model confuse water with urban (or vice versa)? These
two classes have different spectral signatures in most features, but the
turbid water and dark urban sub-populations create genuine overlap.
The confusion matrix makes this visible.

Use the model trained with your chosen `max_depth` from Section 5.1.

In [ ]:
# Listing 8.
# TODO: Evaluate the final model.
#
# 1. Generate predictions on the test set using the model from Part C.
#
# 2. Print overall accuracy.
#
# 3. Print the classification report with target_names=CLASS_NAMES.
#    Focus on water (class 2): what is its recall? How many water pixels
#    is the model missing, and which class does it confuse them with?
#
# 4. Plot the confusion matrix using ConfusionMatrixDisplay.from_predictions().
#    Use display_labels=CLASS_NAMES.
#    Examine the off-diagonal entries: which pair of classes is most often
#    confused?

# YOUR CODE HERE

### 5.3 Interpretation

Decision trees offer two forms of direct interpretability: the tree
structure itself (which feature is used at each split, and at what
threshold), and the feature importances (how much each feature reduces
impurity across all splits where it is used).

Feature importances in scikit-learn are computed as the total weighted
Gini impurity reduction attributable to each feature across all nodes in
the tree. They sum to 1. Be aware that correlated features share
importance: if `ndvi` and `ndwi` carry the same information, the tree
may split the importance arbitrarily between them depending on which
threshold it encountered first during training.

In [ ]:
# Listing 9.
# TODO: Inspect feature importances.
#
# 1. Extract .feature_importances_ from the fitted model.
#    Create a pandas Series with feature names as the index.
#    Sort by importance (descending) and print.
#
# 2. Plot a horizontal bar chart of feature importances, sorted so the
#    most important feature is at the top.
#
# 3. Do the top features match what you would expect from the physical
#    descriptions and the KDE plots in Section 4.3?
#
# 4. If ndvi and ndwi both appear in the top features, consider: are
#    they contributing independently or splitting credit for the same
#    underlying signal (nir_ref)?

# YOUR CODE HERE

In [ ]:
# Listing 10.
# TODO: Visualise the tree structure.
#
# Plot the first three levels of your fitted tree using plot_tree().
# Use these arguments for readability:
#   - max_depth=3  (show only the top 3 levels)
#   - feature_names=feature_cols  (label the splits)
#   - class_names=CLASS_NAMES
#   - filled=True   (colour nodes by majority class)
#   - impurity=True (show Gini impurity at each node)
#   - rounded=True
#   - fontsize=9
# Set the figure size large enough to read: figsize=(20, 8).
#
# Look at the root node: which feature does the tree split on first?
# Does this match the most important feature from the importances plot?
# Read the first two levels: can you describe the decision logic in
# plain English?

# YOUR CODE HERE

### 5.4 Reflection Questions

1. In the depth sweep from Section 5.1, the test accuracy likely peaks
   at a relatively shallow depth (3-6) and then decreases. Explain this
   in terms of bias and variance: what is happening to the tree as depth
   increases beyond the optimum?

2. Water is the smallest class (~10%) and the one with the lowest recall.
   If you trained the tree with `class_weight='balanced'`, how would this
   change the training process, and what effect would you expect on water
   recall and on the precision of the other classes?

3. `ndvi` and `ndwi` are both derived from `nir_ref`. If you removed both
   indices and kept only the four raw band reflectances plus texture and
   thermal, do you expect performance to drop significantly? How would you
   test this?

4. The `texture_var` feature captures spatial heterogeneity within a small
   window. At coarser spatial resolutions (e.g. 250-metre MODIS pixels
   instead of 10-metre Sentinel-2), texture becomes meaningless because
   every pixel is already a mixture of surface types. What would you expect
   to happen to texture importance if the dataset were generated at
   coarser resolution?

5. A decision tree with `max_depth=3` has at most 8 leaf nodes. Given that
   there are 15,000 training samples and three classes, do you think 8
   regions in feature space are enough to capture the real class boundaries,
   or are you accepting some underfitting in exchange for better
   generalisation?

---
## 6. Solution

**Read this section only after attempting the challenge yourself.**
Every code cell is followed by an explanation of the reasoning, not just
a description of what the code does. The cells are self-contained and
run from top to bottom without depending on any variables from Section 5.

### Step 1: Preprocessing

No feature scaling is applied. A decision tree splits on a single feature
at a threshold: it compares the value of one feature against a constant.
Multiplying all values of a feature by a constant (as StandardScaler does)
shifts the optimal threshold by the same factor but does not change which
split is chosen or how much impurity it reduces. Scaling is therefore
unnecessary, unlike for gradient-based methods where it affects convergence.

Stratification is used because the water class has only ~1,690 samples.
Without it, the random split could place significantly fewer water samples
in either the training or test set by chance, distorting both the learned
decision boundaries and the test metrics.

In [ ]:
# Listing 11.
feature_cols = ['blue_ref', 'red_ref', 'nir_ref', 'swir_ref',
                'ndvi', 'ndwi', 'texture_var', 'thermal_bt']

X = df[feature_cols].values
y = df['land_cover'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set:  X={X_train.shape}, y={y_train.shape}')
print(f'Test set:      X={X_test.shape},  y={y_test.shape}')
print()
print('Class counts in training set:')
for cls in [0, 1, 2]:
    n = np.sum(y_train == cls)
    print(f'  {CLASS_NAMES[cls]:12s}: {n:5d}  ({100*n/len(y_train):.1f}%)')
print('Class counts in test set:')
for cls in [0, 1, 2]:
    n = np.sum(y_test == cls)
    print(f'  {CLASS_NAMES[cls]:12s}: {n:5d}  ({100*n/len(y_test):.1f}%)')

### Step 2: Training and Depth Selection

We first train an unconstrained tree to establish the overfitting baseline,
then sweep depths to find where test accuracy peaks.

The unconstrained tree fits every training sample: training accuracy will
be 1.0 (or very close). The test accuracy will be noticeably lower because
the tree has memorised the ~4% label noise and the mixed-pixel ambiguities
rather than learning the underlying spectral patterns.

In [ ]:
# Listing 12.
# Unconstrained tree: establish overfitting baseline
dt_full = DecisionTreeClassifier(random_state=42)
dt_full.fit(X_train, y_train)

train_acc_full = accuracy_score(y_train, dt_full.predict(X_train))
test_acc_full  = accuracy_score(y_test,  dt_full.predict(X_test))

print(f'Unconstrained tree depth: {dt_full.get_depth()}')
print(f'Training accuracy: {train_acc_full:.4f}')
print(f'Test accuracy:     {test_acc_full:.4f}')
print(f'Overfitting gap:   {train_acc_full - test_acc_full:.4f}')

The overfitting gap is clearly visible: the unconstrained tree fits the
training data nearly perfectly but generalises worse than a shallower tree.
This is the classic decision tree bias-variance trade-off. Every additional
level of depth allows the tree to respond to smaller and smaller differences
in the training data, eventually responding to noise rather than signal.

In [ ]:
# Listing 13.
# Depth sweep: find the optimal max_depth
depths = range(1, 16)
train_accs = []
test_accs  = []

for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, dt.predict(X_train)))
    test_accs.append(accuracy_score(y_test,  dt.predict(X_test)))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(depths, train_accs, 'o-', label='Training accuracy',  color='steelblue',  linewidth=2)
ax.plot(depths, test_accs,  's-', label='Test accuracy',      color='firebrick',  linewidth=2)
ax.axvline(depths[np.argmax(test_accs)], color='grey', linestyle='--', linewidth=1,
           label=f'Best depth = {depths[np.argmax(test_accs)]}')
ax.set_xlabel('max_depth', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Training vs Test Accuracy by Tree Depth (RobSat)', fontsize=13)
ax.legend(fontsize=11)
ax.set_xticks(list(depths))
plt.tight_layout()
plt.show()

best_depth = depths[np.argmax(test_accs)]
print(f'Best test accuracy: {max(test_accs):.4f} at max_depth={best_depth}')

The plot reveals the characteristic shape of the bias-variance trade-off
for decision trees:

- At very shallow depths (1-2 nodes), the tree underfits: it cannot capture
  the multi-feature structure needed to separate three overlapping classes,
  and both training and test accuracy are low.
- As depth increases from 2 to the optimum, both training and test accuracy
  rise because the tree is learning genuine spectral patterns.
- Beyond the optimum, training accuracy continues to rise but test accuracy
  plateaus or falls: the tree is now memorising training noise.

The optimal depth for this dataset is shallow by the standards of what a
tree *could* grow to. This is expected for a noisy dataset with overlapping
classes: the signal is strong enough to be captured in a few splits, but
the noise makes deeper splits counterproductive.

In [ ]:
# Listing 14.
# Final model at the chosen depth
dt_final = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
dt_final.fit(X_train, y_train)

print(f'Final model: max_depth={best_depth}, actual depth={dt_final.get_depth()}')

### Step 3: Evaluating the Final Model

The class distribution (60:30:10) means a naive classifier that always
predicted urban would achieve ~59% accuracy. Overall accuracy above this
baseline is meaningful, but per-class recall for the water class is the
most operationally important metric: missing water pixels in a land cover
product leads to errors in flood mapping, water resource monitoring, and
coastal change detection.

In [ ]:
# Listing 15.
y_pred = dt_final.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f'Overall accuracy: {acc:.4f}\n')
print('Classification report:')
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

The classification report shows that urban and vegetation are classified
well. Water recall is the weakest metric, reflecting the turbid-water and
dark-urban overlaps built into the dataset. These are not fixable by
changing the tree depth alone: they reflect genuine spectral ambiguity
in the feature space.

The gap between water precision and recall reveals which direction the
errors go. If recall is lower than precision, the model is missing water
pixels (classifying them as something else). If precision is lower than
recall, the model is falsely flagging non-water pixels as water. Both
have different implications in a mapping context.

In [ ]:
# Listing 16.
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=CLASS_NAMES,
    cmap='Blues',
    ax=ax
)
ax.set_title(f'Confusion Matrix: RobSat Land Cover\n(DecisionTree, max_depth={best_depth})',
             fontsize=10)
plt.tight_layout()
plt.show()

The confusion matrix shows the absolute counts of correct and incorrect
predictions. The main off-diagonal entries are almost always in the
urban-water pair: dark urban surfaces and turbid water are the primary
confusion point, as expected from the feature distributions. The
urban-vegetation confusion is smaller but non-zero, driven by the
mixed urban-vegetation pixels introduced during data generation.

Vegetation and water are rarely confused with each other, which makes
physical sense: vegetation has high NIR and low NDWI; water has near-zero
NIR and positive NDWI. These are strong, orthogonal signals.

### Step 4: Feature Importances

In [ ]:
# Listing 17.
importances = pd.Series(dt_final.feature_importances_, index=feature_cols)
importances_sorted = importances.sort_values(ascending=False)

print('Feature importances (Gini impurity reduction, summing to 1):')
print(importances_sorted.round(4).to_string())
print()

fig, ax = plt.subplots(figsize=(8, 5))
importances_sorted.sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Feature Importance (Gini)', fontsize=11)
ax.set_title('Decision Tree Feature Importances: RobSat', fontsize=12)
ax.axvline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

The importance ranking reflects the physics. Features related to the
NIR band (`ndvi`, `nir_ref`, `ndwi`) typically dominate because NIR
is the most discriminating wavelength across all three classes:
vegetation is bright, water is dark, urban is intermediate.

Note that `ndvi` and `ndwi` are both computed from `nir_ref` and
will tend to share the importance that would otherwise go entirely to
NIR if only the raw band were present. Their individual importance
scores therefore understate the total contribution of the NIR signal.

`thermal_bt` and `texture_var` typically receive modest but non-zero
importance. Thermal adds discriminating power for the urban-vegetation
boundary (urban heat island) but overlaps too much with the spectral
features to dominate. Texture helps separate heterogeneous urban
surfaces from smooth water, complementing the spectral information.

`swir_ref` and the visible band reflectances (`blue_ref`, `red_ref`)
are less important at shallow depths because the tree resolves most
ambiguity through NIR-derived splits first.

### Step 5: Visualising the Tree

Plotting the first three levels shows the decision logic the tree has
learned. At shallow depths this is directly interpretable in terms of
spectral thresholds.

In [ ]:
# Listing 18.
fig, ax = plt.subplots(figsize=(22, 8))
plot_tree(
    dt_final,
    max_depth=3,
    feature_names=feature_cols,
    class_names=CLASS_NAMES,
    filled=True,
    impurity=True,
    rounded=True,
    fontsize=9,
    ax=ax
)
ax.set_title(f'Decision Tree (first 3 levels shown): RobSat Land Cover', fontsize=12)
plt.tight_layout()
plt.show()

Reading the tree from the root:

- The root split is almost certainly on an NIR-related feature (NDVI,
  NDWI, or `nir_ref` directly). A high NDVI threshold routes most
  vegetation pixels to the left; a low threshold routes urban and water
  pixels to the right.
- The second level splits separate the remaining class confusion. On the
  high-NDVI side, a secondary split may refine the vegetation-urban
  boundary for sparse vegetation. On the low-NDVI side, a split on thermal
  or SWIR separates most urban from water.
- By depth 3, the most common and easily separated patterns are handled.
  Deeper splits address progressively smaller and noisier sub-populations,
  which is why they improve training accuracy without consistently improving
  test accuracy.

---
## 7. References

1. Breiman, L., Friedman, J., Olshen, R.A., and Stone, C.J. (1984).
   *Classification and Regression Trees*. Wadsworth & Brooks/Cole.
   The foundational reference for CART, the algorithm implemented in
   scikit-learn's `DecisionTreeClassifier`.

2. McFeeters, S.K. (1996). The use of the Normalised Difference Water Index
   (NDWI) in the delineation of open water features.
   *International Journal of Remote Sensing*, 17(7), 1425-1432.
   https://doi.org/10.1080/01431169608948714
   Original paper defining the NDWI feature used in this dataset.

3. Drusch, M. et al. (2012). Sentinel-2: ESA's Optical High-Resolution
   Mission for GMES Operational Services.
   *Remote Sensing of Environment*, 120, 25-36.
   https://doi.org/10.1016/j.rse.2011.11.026
   Reference for the Sentinel-2 sensor whose spectral bands inspired
   the features in the RobSat dataset.

4. ESA Climate Change Initiative Land Cover. (2017). Product User Guide
   Version 2.0. UCLouvain, Belgium.
   https://www.esa-landcover-cci.org
   Authoritative source for global land cover classification methodology
   and accuracy assessment standards.

5. Pedregosa, F. et al. (2011). Scikit-learn: Machine Learning in Python.
   *Journal of Machine Learning Research*, 12, 2825-2830.
   https://jmlr.org/papers/v12/pedregosa11a.html
   Cite this when reporting results obtained with scikit-learn.